In [0]:
dbutils.widgets.dropdown("source_system", "allscripts_scm", ["allscripts_scm", "allscripts_tw", "epic_clarity"], label="1. Select Source System")
dbutils.widgets.text("fully_qualified_table_name", "<database>.<schema>.<table>", label="2. Fully Qualified Table Name")
dbutils.widgets.text("table_name", "<table>", label="3. Enter Table Name")
dbutils.widgets.text("source_id", "<id>", label="4. Enter Source ID")
dbutils.widgets.text("source_value", "<value>", label="5. Enter Source Value")


In [0]:
# =============================================================================
# QO → OMOP MAPPING PIPELINE (Fixed for Serverless)
# =============================================================================
# Key fix: Replaced toPandas() of entire OMOP concept table with pure-Spark
# token-based candidate retrieval via join. No data collected to driver.
# =============================================================================

import pyspark.sql.functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)
from pyspark.sql import Window
from datetime import datetime

# --- Table names ---
SOURCE_TABLE = dbutils.widgets.get("fully_qualified_table_name") #<<<<UPDATE
OMOP_TABLE = "_exponent.omop.concept"
table_name = dbutils.widgets.get("table_name")
source_id = dbutils.widgets.get("source_id")
source_value = dbutils.widgets.get("source_value")
DISTINCT_RESULTS_TABLE = "_exponent.results_store.distinct_results_" + table_name
FINAL_ROW_RESULTS_TABLE = "_exponent.results_store.final_row_" + table_name
REVIEW_TABLE = "_exponent.results_store.review_" + table_name
FINAL_OUTPUT_TABLE = "_exponent.results_store.final_output_" + table_name

# --- Config ---
TOP_N = 10
AUTO_MATCH_THRESHOLD = 0.90
REVIEW_THRESHOLD = 0.60
MINIMUM_SCORE_THRESHOLD = 0.05

RUN_ID = f"QO_OMOP_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}"
RUN_TIMESTAMP = datetime.utcnow().isoformat(timespec="seconds")

# --- Explicit schema for distinct mapping results ---
RESULT_SCHEMA = StructType([
    StructField("run_id", StringType(), True),
    StructField("source_id", StringType(), True),
    StructField("source_value", StringType(), True),
    StructField("expanded_source_value", StringType(), True),
    StructField("candidate_count", IntegerType(), True),
    StructField("best_concept_id", StringType(), True),
    StructField("best_concept_name", StringType(), True),
    StructField("best_domain_id", StringType(), True),
    StructField("match_status", StringType(), True),
    StructField("final_status", StringType(), True),
    StructField("confidence", DoubleType(), True),
    StructField("reason", StringType(), True),
    StructField("domain_hint", StringType(), True),
    StructField("mapping_method", StringType(), True),
])

print(f"Run ID   : {RUN_ID}")
print(f"Timestamp: {RUN_TIMESTAMP}")
print("Constants loaded.")

In [0]:
# =============================================================================
# RESET OUTPUT TABLES (run for clean rerun)
# =============================================================================
for tbl in [
    DISTINCT_RESULTS_TABLE,
    FINAL_ROW_RESULTS_TABLE,
    REVIEW_TABLE,
    FINAL_OUTPUT_TABLE,
]:
    spark.sql(f"DROP TABLE IF EXISTS {tbl}")
    print(f"  Dropped {tbl}")
print("All output tables dropped.")

In [0]:
# =============================================================================
# LOAD AND CLEAN SOURCE DATA
# =============================================================================
df_source = (
    spark.table(SOURCE_TABLE)
    .select(
        F.col(source_id).cast("bigint").cast("string").alias("source_id"), #<<<<<<UPDATE
        F.col(source_value).alias("source_value"),  #<<<<<<UPDATE
    )
    .filter(F.col("source_value").isNotNull())
    .withColumn(
        "source_value_clean",
        F.lower(F.trim(F.col("source_value")))
    )
)

_source_total = df_source.count()
print(f"Source rows loaded: {_source_total:,}")
display(df_source.limit(5))

In [0]:
# =============================================================================
# LOAD AND FILTER OMOP CONCEPTS
# =============================================================================
# Filter to standard + valid concepts only (~3.5M vs ~10M full table).
# This is the key performance fix: we never collect OMOP to pandas.
df_omop = (
    spark.table(OMOP_TABLE)
    .filter("standard_concept = 'S' AND invalid_reason IS NULL")
    .select("concept_id", "concept_name", "domain_id", "vocabulary_id", "concept_class_id")
    .filter(F.col("concept_name").isNotNull())
    .withColumn("concept_name_lower", F.lower(F.trim(F.col("concept_name"))))
)

_omop_count = df_omop.count()
print(f"OMOP concepts (standard + valid): {_omop_count:,}")
display(df_omop.limit(5))

In [0]:
# =============================================================================
# TOKENIZE + BUILD CANDIDATES VIA SPARK TOKEN JOIN
# =============================================================================
# Tokenize both sides in Spark and join on shared tokens.
# Nothing is collected to the driver.
# IMPROVED: Added pre-filter for non-clinical text + domain inference.

# --- Distinct source values ---
df_distinct = (
    df_source
    .groupBy("source_value_clean")
    .agg(F.first("source_value").alias("source_value"))
)
total_distinct = df_distinct.count()
print(f"Distinct source values to map: {total_distinct:,}")

# ---- PRE-FILTER: reject non-clinical entries ----
BAD_INPUT_PATTERN = r"\b(comment|visit|form|question|survey)\b"

df_distinct_rejected = df_distinct.filter(
    F.col("source_value_clean").rlike(BAD_INPUT_PATTERN)
)
df_distinct_clinical = df_distinct.filter(
    ~F.col("source_value_clean").rlike(BAD_INPUT_PATTERN)
)

total_rejected = df_distinct_rejected.count()
total_clinical = df_distinct_clinical.count()
print(f"Pre-filtered (rejected non-clinical): {total_rejected:,}")
print(f"Clinical values (to map): {total_clinical:,}")

# ---- DOMAIN INFERENCE from source_value ----
LAB_PATTERN = (
    r"\b(hgb|hb|ldl|hdl|t3|t4|tsh|a1c|hba1c|glucose|protein|cholesterol"
    r"|triglyceride|creatinine|bun|sodium|potassium|calcium|albumin"
    r"|bilirubin|hemoglobin|hematocrit|platelet|wbc|rbc|iron|ferritin"
    r"|folate|vitamin|lipase|amylase|troponin|bnp|psa|cea|afp|esr|crp"
    r"|alt|ast|gfr|inr|ptt|fibrinogen|cortisol|insulin|magnesium"
    r"|phosphorus|uric|urea)\b"
)
IMAGING_PATTERN = (
    r"\b(mammo|mammograph|sonogram|ultrasound|scan|xray|x-ray"
    r"|mri|imaging|radiograph|fluoroscopy|angiograph|echocardiogram|dexa)\b"
)
QUESTION_PATTERN = (
    r"(\?|\bdo you\b|\bare you\b|\bhave you\b|\bdoes the\b"
    r"|\bis the\b|\bhow often\b|\bhow many\b|\bhow much\b)"
)
ADMIN_PATTERN = r"\b(case worker|scheduling|appointment|billing|registration|intake)\b"

df_distinct_clinical = df_distinct_clinical.withColumn(
    "domain_hint",
    F.when(F.col("source_value_clean").rlike(ADMIN_PATTERN), F.lit("REJECT"))
     .when(F.col("source_value_clean").rlike(LAB_PATTERN), F.lit("Measurement"))
     .when(F.col("source_value_clean").rlike(IMAGING_PATTERN), F.lit("Procedure"))
     .when(F.col("source_value_clean").rlike(QUESTION_PATTERN), F.lit("Observation"))
     .otherwise(F.lit(None).cast(StringType()))
)

# --- Tokenize source values ---
df_src_tokens = (
    df_distinct_clinical
    .withColumn(
        "src_tokens_arr",
        F.split(F.regexp_replace(F.col("source_value_clean"), "[^a-z0-9]+", " "), "\\s+")
    )
    .withColumn("src_token_count", F.size("src_tokens_arr"))
    .select(
        "source_value_clean", "source_value", "src_token_count", "domain_hint",
        F.explode("src_tokens_arr").alias("token")
    )
    .filter((F.length("token") > 1) & (F.col("token") != ""))
)

# --- Tokenize OMOP concept names ---
df_omop_tokens = (
    df_omop
    .withColumn(
        "omop_tokens_arr",
        F.split(F.regexp_replace(F.col("concept_name_lower"), "[^a-z0-9]+", " "), "\\s+")
    )
    .select(
        "concept_id", "concept_name", "concept_name_lower",
        "domain_id", "vocabulary_id", "concept_class_id",
        F.explode("omop_tokens_arr").alias("token")
    )
    .filter((F.length("token") > 1) & (F.col("token") != ""))
)

# --- Join on shared tokens to get candidate pairs ---
df_candidates = (
    df_src_tokens
    .join(df_omop_tokens, "token", "inner")
    .groupBy(
        "source_value_clean", "source_value", "src_token_count", "domain_hint",
        "concept_id", "concept_name", "concept_name_lower",
        "domain_id", "vocabulary_id", "concept_class_id"
    )
    .agg(F.countDistinct("token").alias("overlap_count"))
)

print("Candidate pairs built via Spark token join (with pre-filter + domain inference).")

In [0]:
# =============================================================================
# SCORE AND RANK CANDIDATES (DETERMINISTIC MAPPING LOGIC)
# =============================================================================
# Scoring rules (preserved):
#   Exact match            = 1.0
#   Partial match (substr) = 0.8
#   Token overlap          = 0.5
# ADJUSTED: Soft domain prioritization (boost ranking, not eliminate).
# No hard confidence cutoff — NO_MATCH only when 0 candidates.

df_scored = (
    df_candidates
    .withColumn(
        "score",
        F.when(
            F.col("concept_name_lower") == F.col("source_value_clean"), F.lit(1.0)
        ).when(
            F.col("concept_name_lower").contains(F.col("source_value_clean"))
            | F.col("source_value_clean").contains(F.col("concept_name_lower")),
            F.lit(0.8)
        ).otherwise(F.lit(0.5))
    )
    .withColumn(
        "score_reason",
        F.when(
            F.col("concept_name_lower") == F.col("source_value_clean"), F.lit("Exact match")
        ).when(
            F.col("concept_name_lower").contains(F.col("source_value_clean"))
            | F.col("source_value_clean").contains(F.col("concept_name_lower")),
            F.lit("Partial match")
        ).otherwise(F.lit("Token overlap"))
    )
)

# ---- HARD FILTER: only remove REJECT (admin/business text) ----
# REJECT entries (case worker, scheduling, etc.) get zero candidates → NO_MATCH.
# All other domain hints are soft — used for ranking boost only.
df_scored_active = df_scored.filter(
    F.col("domain_hint").isNull() | (F.col("domain_hint") != "REJECT")
)

# ---- SOFT DOMAIN BOOST: prioritize domain-matching candidates ----
df_scored_active = df_scored_active.withColumn(
    "domain_bonus",
    F.when(
        F.col("domain_hint").isNotNull() & (F.col("domain_id") == F.col("domain_hint")),
        F.lit(1)
    ).otherwise(F.lit(0))
)

# --- Rank: domain bonus → score → overlap → concept_id ---
w = Window.partitionBy("source_value_clean").orderBy(
    F.desc("domain_bonus"), F.desc("score"), F.desc("overlap_count"), F.asc("concept_id")
)
df_best = (
    df_scored_active
    .withColumn("rank", F.row_number().over(w))
    .filter(F.col("rank") == 1)
    .drop("rank")
)

# --- Count candidates per source value ---
df_candidate_counts = (
    df_scored_active
    .groupBy("source_value_clean")
    .agg(F.count("*").alias("candidate_count"))
)

# --- Join best match with candidate count ---
df_matched = df_best.join(df_candidate_counts, "source_value_clean", "left")

print("Scoring with soft domain prioritization complete.")

In [0]:
# =============================================================================
# BUILD DISTINCT RESULTS WITH STATUS CLASSIFICATION
# =============================================================================
# ADJUSTED: NO_MATCH only when candidate_count == 0.
# All matched entries kept — low confidence flagged as REVIEW_REQUIRED.

# --- Matched values → classify status ---
df_matched_results = (
    df_matched
    .withColumn(
        "match_status",
        F.lit("MATCH")
    )
    .withColumn(
        "final_status",
        F.when(F.col("score") >= AUTO_MATCH_THRESHOLD, F.lit("AUTO_MATCH"))
        .otherwise(F.lit("REVIEW_REQUIRED"))
    )
    .select(
        F.lit(RUN_ID).alias("run_id"),
        F.col("source_value_clean").alias("source_id"),
        F.col("source_value"),
        F.col("source_value_clean").alias("expanded_source_value"),
        F.col("candidate_count").cast(IntegerType()),
        F.col("concept_id").cast("string").alias("best_concept_id"),
        F.col("concept_name").alias("best_concept_name"),
        F.col("domain_id").alias("best_domain_id"),
        F.col("match_status"),
        F.col("final_status"),
        F.col("score").cast(DoubleType()).alias("confidence"),
        F.col("score_reason").alias("reason"),
        F.col("domain_hint"),
        F.lit("DETERMINISTIC").alias("mapping_method"),
    )
)

# --- Unmatched clinical values (0 candidates OR REJECT domain) ---
df_unmatched = (
    df_distinct_clinical
    .join(df_matched, "source_value_clean", "left_anti")
    .select(
        F.lit(RUN_ID).alias("run_id"),
        F.col("source_value_clean").alias("source_id"),
        F.col("source_value"),
        F.col("source_value_clean").alias("expanded_source_value"),
        F.lit(0).cast(IntegerType()).alias("candidate_count"),
        F.lit(None).cast(StringType()).alias("best_concept_id"),
        F.lit(None).cast(StringType()).alias("best_concept_name"),
        F.lit(None).cast(StringType()).alias("best_domain_id"),
        F.lit("NO_CANDIDATES").alias("match_status"),
        F.lit("NO_MATCH").alias("final_status"),
        F.lit(0.0).cast(DoubleType()).alias("confidence"),
        F.when(F.col("domain_hint") == "REJECT",
               F.lit("Rejected admin/business text"))
         .otherwise(F.lit("No candidates found after token lookup")).alias("reason"),
        F.col("domain_hint"),
        F.lit("DETERMINISTIC").alias("mapping_method"),
    )
)

# --- Pre-filtered non-clinical values (comment, visit, form, question, survey) ---
df_prefiltered = (
    df_distinct_rejected
    .select(
        F.lit(RUN_ID).alias("run_id"),
        F.col("source_value_clean").alias("source_id"),
        F.col("source_value"),
        F.col("source_value_clean").alias("expanded_source_value"),
        F.lit(0).cast(IntegerType()).alias("candidate_count"),
        F.lit(None).cast(StringType()).alias("best_concept_id"),
        F.lit(None).cast(StringType()).alias("best_concept_name"),
        F.lit(None).cast(StringType()).alias("best_domain_id"),
        F.lit("FILTERED").alias("match_status"),
        F.lit("NO_MATCH").alias("final_status"),
        F.lit(0.0).cast(DoubleType()).alias("confidence"),
        F.lit("Filtered non-clinical input").alias("reason"),
        F.lit(None).cast(StringType()).alias("domain_hint"),
        F.lit("DETERMINISTIC").alias("mapping_method"),
    )
)

# --- Combine matched + unmatched + prefiltered ---
df_distinct_results = (
    df_matched_results
    .unionByName(df_unmatched)
    .unionByName(df_prefiltered)
)

_distinct_count = df_distinct_results.count()
print(f"Distinct results: {_distinct_count:,} (expected: {total_distinct:,})")
assert _distinct_count == total_distinct, (
    f"MISMATCH: got {_distinct_count:,} vs expected {total_distinct:,}"
)
print("PASS: All distinct values have a mapping result.")

In [0]:
# =============================================================================
# SAVE DISTINCT RESULTS
# =============================================================================
(
    df_distinct_results
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DISTINCT_RESULTS_TABLE)
)
print(f"Saved → {DISTINCT_RESULTS_TABLE}")
display(df_distinct_results.limit(5))

In [0]:
# =============================================================================
# JOIN DISTINCT RESULTS → ROW-LEVEL + SAVE ALL OUTPUT TABLES
# =============================================================================

# Select only mapping columns from distinct results (avoid column name clashes)
df_mapping = df_distinct_results.select(
    F.col("source_id").alias("mapped_source_value_clean"),
    "expanded_source_value", "candidate_count", "best_concept_id",
    "best_concept_name", "best_domain_id", "match_status", "final_status",
    "confidence", "reason", "domain_hint", "mapping_method", "run_id"
)

df_row_joined = (
    df_source
    .join(
        df_mapping,
        df_source["source_value_clean"] == df_mapping["mapped_source_value_clean"],
        how="left",
    )
    .drop("mapped_source_value_clean")
)

# --- Build final detailed DataFrame ---
df_final = df_row_joined.select(
    F.col("source_id").alias(source_id), #<<<<<<<UPDATE
    F.col("source_value").alias(source_value), #<<<<<<<UPDATE
    F.col("best_concept_id").alias("concept_id"),
    F.col("source_value_clean"),
    F.col("expanded_source_value"),
    F.col("candidate_count").cast(IntegerType()),
    F.col("best_concept_name"),
    F.col("best_domain_id"),
    F.col("confidence").cast(DoubleType()),
    F.col("match_status"),
    F.col("final_status"),
    F.col("reason"),
    F.col("domain_hint"),
    F.col("mapping_method"),
    F.col("run_id"),
)

_final_count = df_final.count()
print(f"Final detailed rows: {_final_count:,} (expected: {_source_total:,})")
assert _final_count == _source_total, (
    f"JOIN MISMATCH: final={_final_count:,} vs source={_source_total:,}"
)
print("PASS: Row-level join count matches source.")

# --- Save detailed results ---
df_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(FINAL_ROW_RESULTS_TABLE)
print(f"Saved → {FINAL_ROW_RESULTS_TABLE}")

# --- Save review queue ---
df_review = df_final.filter(F.col("final_status").isin("REVIEW_REQUIRED", "NO_MATCH"))
df_review.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(REVIEW_TABLE)
print(f"Saved → {REVIEW_TABLE}")

# --- Save clean output: id, entryname, concept_id ---
df_output = df_final.select(source_id, source_value, "concept_id")  #<<<<<<<UPDATE
df_output.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(FINAL_OUTPUT_TABLE)
_output_count = df_output.count()
print(f"Saved → {FINAL_OUTPUT_TABLE}  ({_output_count:,} rows)")
assert _output_count == _source_total, (
    f"OUTPUT MISMATCH: {_output_count:,} vs {_source_total:,}"
)
print(f"PASS: Output row count matches source ({_source_total:,}).")
display(df_output.limit(10))

In [0]:
# =============================================================================
# SUMMARY REPORT
# =============================================================================
print("=" * 60)
print(f"  OMOP MAPPING SUMMARY — {RUN_ID}")
print("=" * 60)

status_summary = df_final.groupBy("final_status").count().orderBy("final_status")
display(status_summary)

auto_match_count = df_final.filter(F.col("final_status") == "AUTO_MATCH").count()
match_rate = auto_match_count / _final_count * 100 if _final_count > 0 else 0

print(f"\nTotal rows          : {_final_count:,}")
print(f"Auto-match rate     : {match_rate:.1f}%")
print(f"Run ID              : {RUN_ID}")
print(f"Method              : Deterministic (Spark token join)")

print("\nDomain breakdown (AUTO_MATCH):")
domain_summary = (
    df_final
    .filter(F.col("final_status") == "AUTO_MATCH")
    .groupBy("best_domain_id")
    .count()
    .orderBy(F.desc("count"))
)
display(domain_summary)

print("\n" + "-" * 60)
print("TABLES PRODUCED:")
print(f"  {DISTINCT_RESULTS_TABLE}")
print(f"  {FINAL_ROW_RESULTS_TABLE}")
print(f"  {REVIEW_TABLE}")
print(f"  {FINAL_OUTPUT_TABLE}")
print("=" * 60)
print("  PIPELINE COMPLETE")
print("=" * 60)

# --- Serverless-safe clear cache ---
try:
    spark.catalog.clearCache()
except Exception as e:
    if "NOT_SUPPORTED_WITH_SERVERLESS" in str(e):
        pass
    else:
        raise

In [0]:
display(spark.table(FINAL_OUTPUT_TABLE).limit(10))

In [0]:
source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
source_id = dbutils.widgets.get("source_id")
source_value = dbutils.widgets.get("source_value")
review_table = REVIEW_TABLE

query = f"""
SELECT
  '{source_system}' AS source_system,
  '{table_name}' AS source_table,
  '{source_value}' AS source_field,
  {review_table}.best_domain_id AS domain_id,
  {review_table}.{source_id} AS source_id,
  {review_table}.{source_value} AS source_value,
  concept_id AS omop_concept_id,
  TRUE AS active_flag,
  CURRENT_TIMESTAMP() AS last_update_tsp
FROM {review_table}
WHERE best_domain_id IS NOT NULL
AND concept_id IS NOT NULL
"""

print(query)
spark.sql(query).display()

# spark.sql(query).createOrReplaceTempView("sample_insert")

In [0]:
# delete_query = f"""
# DELETE FROM _exponent.omop_mapping.domain_source_to_concept
# WHERE source_table = '{table_name}'
# AND LOWER(source_field) = LOWER('{source_value}')
# AND source_id = '{source_id}'
# """
# print(delete_query)
# spark.sql(delete_query)

In [0]:
# source_system = dbutils.widgets.get("source_system")
# table_name = dbutils.widgets.get("table_name")
# source_id = dbutils.widgets.get("source_id")
# source_value = dbutils.widgets.get("source_value")
# review_table = REVIEW_TABLE

# query = f"""
# INSERT INTO _exponent.omop_mapping.domain_source_to_concept
# SELECT
#   '{source_system}' AS source_system,
#   '{table_name}' AS source_table,
#   '{source_value}' AS source_field,
#   {review_table}.best_domain_id AS domain_id,
#   {review_table}.{source_id} AS source_id,
#   {review_table}.{source_value} AS source_value,
#   concept_id AS omop_concept_id,
#   TRUE AS active_flag,
#   CURRENT_TIMESTAMP() AS last_update_tsp

# FROM {review_table}
# WHERE best_domain_id IS NOT NULL
# AND omop_concept_id IS NOT NULL
# """

# print(query)
# spark.sql(query)